# 04 — Commissioning years and naive durations

**Input:** `Results/3.xlsx`
**Output:** `Results/4.xlsx`

Computes the first and last commissioning year declared for each investment,
and from them the naive delay and duration. These are descriptive quantities:
they are biased by left-truncation and right-censoring and are superseded by
the Markov chain estimates in A_1. Two commissioning years misreported in the
2024 cycle are corrected against the promoters' own documentation.


In [1]:
import pandas as pd
import re
import ast

In [2]:
df = pd.read_excel("Results/3.xlsx", header=[0,1], index_col=0)

In [3]:
corrections = {
    1014: 2029,
    1207: 2036
}

for inv_idx, correct_year in corrections.items():
    mask = df[('2024', 'Inv_index')] == inv_idx
    df.loc[mask, ('2024', 'Inv_Commissioning_Year_Standardized')] = correct_year


### Duration 

In [4]:
def fetch_std_comm(row, meta_field):
    year = row[('meta', meta_field)]
    if pd.isna(year):
        return pd.NA
    key = (str(int(year)), 'Inv_Commissioning_Year_Standardized')
    return row.get(key, pd.NA)

df[('meta', 'Inv_first_reported_commissioning_year')] = df.apply(
    lambda r: fetch_std_comm(r, 'Inv_First_year_present'),
    axis=1
)
df[('meta', 'Inv_last_reported_commissioning_year')] = df.apply(
    lambda r: fetch_std_comm(r, 'Inv_Last_year_present'),
    axis=1
)

df[('meta', 'delay')] = (
    df[('meta', 'Inv_last_reported_commissioning_year')]
  - df[('meta', 'Inv_first_reported_commissioning_year')]
)

df[('meta', 'duration')] = (
    df[('meta', 'Inv_last_reported_commissioning_year')]
  - df[('meta', 'Inv_First_year_present')]
)


In [5]:

dur = df[('meta', 'duration')]
delay = df[('meta', 'delay')]

min_d     = dur.min()
max_d     = dur.max()
zero_cnt  = (dur == 0).sum()

dur_for_avg = dur[(dur >= 0) & (dur <= 500)]

median_d  = dur_for_avg.median()
avg_d     = dur_for_avg.mean()

# Compute each summary
min_delay     = delay.min()
max_delay     = delay.max()
zero_count    = (delay == 0).sum()
median_delay  = delay.median()
avg_delay     = delay.mean()
avg_pos_delay = delay[delay > 0].mean()
std_pos_delay = delay[delay > 0].std()
avg_neg_delay = delay[delay < 0].mean()
three_quarter_pos_delay = delay[delay > 0].quantile(0.75)
positive_count_delay = int((delay > 0).sum())
negative_count_delay = int((delay < 0).sum())


# Print out
print(f"Min delay:                          {min_delay}")
print(f"Max delay:                          {max_delay}")
print(f"Number of zeros:                    {zero_count}")
print(f"Median delay:                       {median_delay}")
print(f"Average delay:                      {avg_delay:.2f}")
print(f"Average of positives (>0):          {avg_pos_delay:.2f}")
print(f"Average of negatives (<0):          {avg_neg_delay:.2f}")
print(f"Number of investments with positive delay: {positive_count_delay}")
print(f"Std dev of positive delays: {std_pos_delay:.2f}")
print(f"75* percentile positive delays: {three_quarter_pos_delay}")
print(f"Number of investments with negative delay: {negative_count_delay}")


print(f"Min duration:               {min_d}")
print(f"Max duration:               {max_d}")
print(f"Number of zeros:            {zero_cnt}")
print(f"Median duration:            {median_d}")
print(f"Average duration:           {avg_d:.2f}")




Min delay:                          -12.0
Max delay:                          20.0
Number of zeros:                    421
Median delay:                       0.0
Average delay:                      1.80
Average of positives (>0):          4.66
Average of negatives (<0):          -3.88
Number of investments with positive delay: 398
Std dev of positive delays: 3.45
75* percentile positive delays: 6.0
Number of investments with negative delay: 67
Min duration:               -2.0
Max duration:               30.0
Number of zeros:            8
Median duration:            10.0
Average duration:           10.19


In [6]:
stats = {
    'min_duration':     min_d,
    'max_duration':     max_d,
    'zero_count':       int(zero_cnt),
    'median_duration':  median_d,
    'average_duration': avg_d,

}

pd.DataFrame([stats])

,min_duration,max_duration,zero_count,median_duration,average_duration
0,-2.0,30.0,8,10.0,10.192758


In [7]:
df.to_excel("Results/4.xlsx")